# Evaluation: Laptop vs Colab vergleichen

Nach jedem Lauf liegen **6 Dateien** in `results/` (Trennzeichen **`;`**, UTF-8 mit BOM):

| Datei | Inhalt |
|-------|--------|
| `metrics.csv` | **Wichtigster Vergleich**: Accuracy, Precision, Recall, F1 (overall, gender, emotion, age) |
| `evaluation_results.csv` | Pro Bild: alle Spalten |
| `predictions.csv` | Pro Bild: kompakte Text-Tripletts |
| `confusion_gender.csv` | Verwirrung Gender |
| `confusion_age.csv` | Verwirrung Altersgruppe |
| `confusion_emotion.csv` | Verwirrung Emotion |

**Tipp:** Für die Berichts-Tabelle reicht meist **`metrics.csv` + Zeiten** aus beiden Umgebungen. Die Confusion-CSVs sind Anhang/Screenshot.

Vorgehen:
1. Laptop: Ordner `results/` zippen → z.B. `results_laptop/` entpacken.
2. Colab: gleiche Evaluation laufen lassen → `results/` herunterladen → `results_colab/`.
3. Unten `PATH_LAPTOP` und `PATH_COLAB` setzen (oder in Colab beide hochladen).

In [ ]:
import pandas as pd
from pathlib import Path

# Anpassen: Ordner mit den jeweiligen results-CSVs
PATH_LAPTOP = Path("results_laptop")   # z.B. nach Entpacken
PATH_COLAB = Path("results_colab")

SEP = ";"
ENC = "utf-8-sig"


def load_metrics(folder: Path) -> pd.DataFrame:
    p = folder / "metrics.csv"
    return pd.read_csv(p, sep=SEP, encoding=ENC)


def load_all(folder: Path) -> dict[str, pd.DataFrame]:
    return {
        "metrics": pd.read_csv(folder / "metrics.csv", sep=SEP, encoding=ENC),
        "evaluation_results": pd.read_csv(folder / "evaluation_results.csv", sep=SEP, encoding=ENC),
        "predictions": pd.read_csv(folder / "predictions.csv", sep=SEP, encoding=ENC),
        "confusion_gender": pd.read_csv(folder / "confusion_gender.csv", sep=SEP, encoding=ENC),
        "confusion_age": pd.read_csv(folder / "confusion_age.csv", sep=SEP, encoding=ENC),
        "confusion_emotion": pd.read_csv(folder / "confusion_emotion.csv", sep=SEP, encoding=ENC),
    }

## 1) Kernvergleich: `metrics.csv` nebeneinander

In [ ]:
m_lap = load_metrics(PATH_LAPTOP).assign(env="Laptop")
m_col = load_metrics(PATH_COLAB).assign(env="Colab")

# Eine Tabelle: Zeilen = scope, Spalten = Metrik, zwei Umgebungen
key_cols = ["scope", "accuracy", "precision", "recall", "f1"]
lap = m_lap[key_cols].rename(columns={c: f"{c}_laptop" for c in key_cols if c != "scope"})
col = m_col[key_cols].rename(columns={c: f"{c}_colab" for c in key_cols if c != "scope"})
compare = lap.merge(col, on="scope", how="outer")
compare

## 2) Zeiten ergänzen (manuell oder aus Log)

Colab: z.B. `%%time` um die Evaluation-Zelle oder Summe aus `total_runtime_s` in `metrics.csv` (steht pro Zeile gleich – ist Gesamtzeit des Laufs).

Unten: `mean_inference_time_s` aus `metrics` anzeigen.

In [ ]:
time_cols = ["scope", "mean_inference_time_s", "std_inference_time_s", "total_runtime_s", "n_images"]
display(m_lap[time_cols])
display(m_col[time_cols])

## 3) Optional: Confusion als Zahlenmatrix (ohne Plot)

Erste Spalte oft `true_*`, Rest pred-Counts — direkt lesbar.

In [ ]:
lap_all = load_all(PATH_LAPTOP)
col_all = load_all(PATH_COLAB)

for name in ["confusion_gender", "confusion_age", "confusion_emotion"]:
    print("===", name, "LAPTOP ===")
    display(lap_all[name])
    print("===", name, "COLAB ===")
    display(col_all[name])